<div style="font-size:18px">

**Download this Notebook**

</div>

Click [here](https://github.com/serc-chile/SERCpy/releases/download/operating_conditions_tutorial/OperatingConditions_tutorial.zip) to download this ipynb file and run it locally in your machine.

The notebook comes as part of a zip file along with a csv file that is used to import the meteorological data.

<div style="height:20px;"></div>

# OperatingConditions class tutorial

In [1]:
## importing the classes needed for this tutorial
from sercpy.meteo_profile import MeteoProfile
from sercpy.demand_profile import DemandProfile
from sercpy.operating_conditions import OperatingConditions
import matplotlib.pyplot as plt

## 1. Define the OperatingConditions instance

An OperatingConditions instance is essentially a merger of a MeteoConditions- and a DemandConditions instance.

It also serves the purpose of reducing the (originally minutal) time resolution of the data stored in these two classes, since for most cases a time resolution of one minute is unnecessarily high, and it increases the computation time without significantly improving the accuracy of the calculations.

Thus, only three elements are needed to define an `OperatingConditions` instance:

- A `MeteoProfile` instance.
- A `DemandProfile` instance.
- The time step length, in minutes.

In [2]:
## Defining a MeteoProfile instance
## More details on how to construct instances of this class can be found in the tutorial dedicated to it
meteo_profile = MeteoProfile(
    
    tmy_file_path = "TMY_Santiago.csv",
    latitude = -33.49818458807115,
    longitude = -70.60757631044005,
    tmy_utc_offset = -4,
    elevation = 579,
    Tamb_col_name = "temp",
    ghi_col_name = "ghi",
    dni_col_name = "dni",
    dhi_col_name = "difh",
    
    )

## Defining a DemandProfile instance
## More details on how to construct instances of this class can be found in the tutorial dedicated to it
daily_demand_profile = [
    0.14, 0.13, 0.10, 0.08,
    0.00, 0.00, 0.00, 0.19,
    0.37, 0.68, 0.87, 1.00,
    1.00, 1.00, 1.00, 1.00,
    1.00, 0.83, 0.68, 0.44,
    0.29, 0.17, 0.12, 0.11 ]

monthly_heat_demand = [
    88.93280632411067, 79.0513833992095, 102.76679841897233,
    120.55335968379447, 140.31620553359684, 160.07905138339922,
    183.7944664031621, 181.8181818181818, 138.3399209486166,
    120.55335968379447, 94.86166007905138, 88.93280632411067 ]

demand_1 = DemandProfile(

    daily_demand_profile = daily_demand_profile,
    monthly_heat_demand = monthly_heat_demand,
    heat_demand_units = 'MWh',
    meteo_profile = meteo_profile,
    fluid = 'water',
    T_set = 60,
    T_in = "mains",
    
)

## Establish the time step
# The time step must be an integer number of minutes, and also a divisor of 60.
# This is done so that an integer number of time steps fits within each hour of the year
time_step_min = 6 # minutes

# Define an OperatingConditions instance from the two objects previously constructed and the time step
op_conditions = OperatingConditions(

    meteo_profile = meteo_profile,
    demand_profile = demand_1,
    time_step_min = time_step_min 

    )

Since having a time step of 6 minutes is equivalent to having 10 time steps per hour, the total number of time steps per year is equal to 87,600.

This can be confirmed by accessing the attribute OperatingConditions.time_steps_per_year

In [3]:
print( f"Time steps per year: {op_conditions.time_steps_per_year}" )

Time steps per year: 87600


Other attributes that may be useful to the user are `time_steps_per_day` and `time_steps_per_hour`.

In [4]:
print( f"Time steps per day: {op_conditions.time_steps_per_day}" )
print( f"Time steps per hour: {op_conditions.time_steps_per_hour}" )

Time steps per day: 240
Time steps per hour: 10


The time step assigned by the user can be retrieved by the user with the attribute names `time_step` (time step in seconds) and `time_step_min` (time step in minutes).

In [5]:
print( f"Time step: {op_conditions.time_step} s" )
print( f"Time step: {op_conditions.time_step_min} min" )

Time step: 360 s
Time step: 6 min


## 2. Modifying the time step of an already defined object

When performing simulations of solar energy systems, the user may need to analyze the effect of using different time step lengths on the simulation results.

The time step of an already defined OperatingConditions instance can be easily modified with the method `OperatingConditions.set_time_step`. The same minutal data used at initialization for creating the yearly profiles with the original time step is used again to create a new dataset, with different time resolution.

In [6]:
op_conditions.set_time_step( time_step_min = 20 )

The new attributes of the object are consistent with the time step just set.

In [7]:
print( f"Time step: {op_conditions.time_step_min} min" )
print( f"Time steps per year: {op_conditions.time_steps_per_year}" )
print( f"Time steps per day: {op_conditions.time_steps_per_day}" )
print( f"Time steps per hour: {op_conditions.time_steps_per_hour}" )

Time step: 20 min
Time steps per year: 26280
Time steps per day: 72
Time steps per hour: 3


## 3. Retrieving data from an OperatingConditions instance

The conditions for a certain time step can be retrieved by using the method `OperatingConditions.get_conditions( time_step_number )`.

The argument `time_step_number` must be an integer, and it is interpreted as the index of the desired time step within the year. Therefore, a one-year simulation can be carried out in the following way:

```python
for time_step_number in range( op_conditions.time_steps_per_year ):
    system_operating_conditions = op_conditions.get_conditions( time_step_number )
    time_step_simulation_results = solar_thermal_system.compute_time_step( system_operating_conditions )
```
<br>

Values of `time_step_number` outside of the range `0 <= time_step_number < time_steps_per_year` are valid as well: in this case, the conditions returned by the `OperatingConditions` instance become cyclic.

In [8]:
## Get the conditions for the 30th time step of the year.
## Since a time step of 20 minutes was just set, this corresponds to 10:00 a.m. on January 1st

print( op_conditions.get_conditions ( 30 ) )

{'timestamp': Timestamp('2025-01-01 10:00:00-0300', tz='America/Santiago'), 'GHI': 605.3288333333333, 'DNI': 910.2184166666666, 'DHI': 76.52233333333334, 'Tamb': 21.311333333333334, 'Tmains': 21.413572278799435, 'flowrate': 1.415065659903934, 'demanded_power': 225012.19938939478, 'T_in': 21.995257266953367, 'T_set': 60.0, 'time_step': 1200}


## 4. Including the solar position data when initializing the object

The standard simulation scheme of **SERCpy** does not need solar position data, since POA<sup>1</sup> irradiance and IAM<sup>2</sup> factors are computed by `MeteoProfile` instances before the simulation is carried out. Therefore, the default behavior of `OperatingConditions` instances is to skip the solar position data stored by the `MeteoProfile` instance, thus reducing the computational cost. The skipped data includes: `"apparent_zenith"`, `"zenith"`, and `"azimuth"`.

This behavior can be changed by the user by setting the initialization argument `include_solar_position` to `True`. The default value of this argument is `False`.

<sup>1</sup> Plane of array  
<sup>2</sup> Incidence angle modifier

In [9]:
# When defining the OperatingConditions instance above, the argument 'include_solar_position' was not specified

# Therefore, trying to access solar position data raises an error

conditions_at_time_step_30  = op_conditions.get_conditions ( 30 )
print( conditions_at_time_step_30[ "apparent_zenith" ] )

KeyError: 'apparent_zenith'

In [10]:
# A new OperatingConditions instance is constructed.
# This time, 'include_solar_position' is set to True
op_conditions = OperatingConditions(

    meteo_profile = meteo_profile,
    demand_profile = demand_1,
    time_step_min = time_step_min,
    include_solar_position = True, )

# Change time step to 20 min
op_conditions.set_time_step( time_step_min = 20 )

# Now, the data includes solar position data
conditions_at_time_step_30  = op_conditions.get_conditions ( 30 )
print( conditions_at_time_step_30[ "apparent_zenith" ] )

48.31106207006495
